# 7.1 Discriminant functions

In the previous chapter, we explored how to describe an image object or *region* by a feature vector $\mathbf{x} = [x_1, x_2, \ldots, x_N]^T \in \mathbb{R}^N$, being $N$ the number of features. In this chapter, our goal is **to classify the object as belonging to one of $M$ object classes or categories** $C_1, \ldots, C_M$, according to said feature vector $\mathbf{x}$. In other words, we pursue the design of a classifier that maps each feature vector $\mathbf{x}$ to a predicted class $\hat{C}(\mathbf{x}) \in \{C_1,\dots, C_M\}$ (see Fig. 1). $\\[15pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/object_recognition_coco.png" width="500"/>
    <figcaption>Fig. 1. Example of object classification within the COCO dataset.</figcaption>
</center>$\\[5pt]$

In this scenario, a computer vision system is in charge of classifying the objects within kitchens. The considered categories are: $C_1=\text{fridge}$, $C_2=\text{bowl}$, $C_3=\text{microwave}$, $C_4=\text{oven}$, $C_5=\text{toaster}$ and $C_6=\text{spoon}$. The results of the object classification are:

$$
\begin{array}{l}
\mathbf{x}^{(1)} = [x^{(1)}_1, x^{(1)}_2, \ldots, x^{(1)}_N]^T \rightarrow \hat{C}(\mathbf{x}^{(1)}) = C_4 \\
\mathbf{x}^{(2)} = [x^{(2)}_1, x^{(2)}_2, \ldots, x^{(2)}_N]^T \rightarrow \hat{C}(\mathbf{x}^{(2)}) = C_1 \\
\mathbf{x}^{(3)} = [x^{(3)}_1, x^{(3)}_2, \ldots, x^{(3)}_N]^T \rightarrow \hat{C}(\mathbf{x}^{(3)}) = C_3 \\
\mathbf{x}^{(4)} = [x^{(4)}_1, x^{(4)}_2, \ldots, x^{(4)}_N]^T \rightarrow \hat{C}(\mathbf{x}^{(4)}) = C_5
\end{array}
$$

For that, the classifier partitions the space defined by the features in $\mathbf{x}$, also referred as **feature space**, into **decision regions** $R_k$, one associated with each class $C_k$, so that if $\mathbf{x} \in R_k$, then $\hat{C}(\mathbf{x}) = C_k$. Fig. 2 shows an example of those regions in a scenario with $M=3$ classes and $N=2$ features, $\mathbf{x}=[x_1,x_2]^T$:
$\\[10pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/decision_regions.png" width="400"/>
    <figcaption>Fig. 2. Example of a feature space with three decision regions, one for each class.</figcaption>
</center>$\\[5pt]$

Two main steps are needed in order to build an object recognition system and work with it:

- A **training (design) phase**, where sample vectors of known objects are used to learn the classifier (supervised learning).
- A **prediction (online) phase**, where the image objects are classified as belonging to one of the classes based on the learned prediction model.

There are two main approaches to build such a classifier:

- **Generative approach:**
  - It models how each class generates the observations, that is, the **class-conditional densities** $p(\mathbf{x}|C_k)$ and the **prior probabilities** $P(C_k)$, whose parameters are **learned from known objects**.
  - Then, the Bayes' rule gives the posterior probabilities $P(C_k|\mathbf{x})$, and $\mathbf{x}$ is assigned to the class with the highest one.
  - Examples: Bayesian classifiers, like the Gaussian or the Naïve Bayes ones.
- **Discriminative approach:**
  - No model of how the observations are generated is built. Instead, the **discriminant functions** $d_k(\mathbf{x})$ (or the posteriors $P(C_k|\mathbf{x})$) are learned directly from labeled data.
  - $\mathbf{x}$ is assigned to the class with the highest score.
  - Examples: Perceptron, Logistic Regression, Support Vector Machines, AdaBoost, Neural Networks.

Fig. 3 summarizes both approaches, which we will explore along this chapter:$\\[10pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/classification_approaches.png" width="800"/>
    <figcaption>Fig. 3. Generative and discriminative approaches to classify a feature vector $\mathbf{x}$.</figcaption>
</center>$\\[5pt]$

This notebook focuses on discriminant functions, which assign a score to each class. Concretely it covers:

- Linear discriminant functions (<a href="#711">section 7.1.1</a>).
- The concept of separability (<a href="#712">section 7.1.2</a>).
- (Optional) Generalized discriminant functions (<a href="#713">section 7.1.3</a>).

All these methods are under the Machine Learning (ML) umbrella. If you want to give a try to Deep Learning (DL) ones, you can do it in notebook *7.5 Deep Learning methods for object detection*!

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> Generative vs discriminative classifiers</font>  
> **Suggested prompt**  
> *Explain, with an everyday example, the difference between a generative and a discriminative classifier, and give one advantage and one drawback of each approach.*

## Problem context - Traffic sign recognition

Before self-driving vehicles can truly operate autonomously, they would need to identify traffic signs like "speed limit", "school ahead" or "turn ahead". This problem is called **Traffic Sign Recognition (TSR)** and is faced by companies like [Tesla](https://www.tesla.com/autopilot) or [Waymo](https://waymo.com/) (from Google), and now by *AliquindoiCars*, a startup located in the Andalusia Technology Park in Málaga.

TSR is part of the features collectively called *Advanced Driver Assistance Systems (ADAS)*. It uses image processing techniques to detect the traffic signs. The detection methods used can be generally divided into color based, shape based, or those using low level features. $\\[3pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/signs.jpg" width="400"/>
    <figcaption>Fig. 4. Examples of traffic signs that a TSR system must recognize.</figcaption>
</center>$\\[5pt]$

Given your growing experience in computer vision, *AliquindoiCars* contacted you asking for a TSR technique to be integrated into a self-driving car. 

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> Traffic sign recognition in practice</font>  
> **Suggested prompt**  
> *In the context of Advanced Driver Assistance Systems, how does a traffic sign recognition system work, from detecting a sign in the image to recognizing its class? In which situations are classical approaches, based on handcrafted features like the shape of the sign and simple classifiers, still a good choice?*

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 07. Object Recognition'
except ImportError:
    pass

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (8.0, 8.0)

images_path = './images/'

### The data

Since most traffic signs in Spain (and in the rest of the world) have a rectangular, circular or triangular shape, they provided you 3 folders containing 20 images each;

- `./images/circles/{0-19}.png`
- `./images/squares/{0-19}.png`
- `./images/triangles/{0-19}.png`

These folders contain segmented and binarized images of traffic signs captured by a camera in a car. **Our task is to design a recognition system able to distinguish each type of shape**.

Let's take a look at some of these images:

In [ ]:
# Load images
img_circle = cv2.imread(images_path + "circles/0.png", 0)
img_square = cv2.imread(images_path + "squares/0.png", 0)
img_triangle = cv2.imread(images_path + "triangles/0.png", 0)

# Show them!
plt.subplot(131)
plt.imshow(img_circle,cmap='gray');
plt.title('Circular shape')

plt.subplot(132)
plt.imshow(img_square,cmap='gray');
plt.title('Rectangular shape')

plt.subplot(133)
plt.imshow(img_triangle,cmap='gray');
plt.title('Triangular shape');

### Computing feature vectors

Prior to describe the methods that will be used to perform recognition, let's build the feature vectors they are going to use. For this, and given your experience, you are going to use the Hu Moments, so each sign will be described by a feature vector with $N=7$ features, $\mathbf{x} = [x_1, x_2, \ldots, x_7]^T$, being $x_f$ its $f$-th Hu moment.

### **<span style="color:green"><b><i>ASSIGNMENT 1: Computing the features</i></b></span>**

**Your first task is** to obtain a $20\times7$ matrix for each type of shape, where rows index images and columns Hu moments. In this way, the 5th row in the matrix characterizing rectangular signs would contain the Hu moments computed from the 5th image with that shape. 

In order to get a first-hand impression of how these features are distributed in the feature space for the different shapes, plot the results using [`plt.scatter()`](https://matplotlib.org/3.2.1/api/_as_gen/matplotlib.pyplot.scatter.html) (show only the first 2 Hu moments).

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Scatter plots with pyplot</font>  
> **Suggested prompt**  
> *I have three NumPy arrays with shape (20, 7), with the Hu moments of the images of three classes of shapes. I need to plot the first two Hu moments of each class in the same scatter plot with Matplotlib's `plt.scatter()`, using a different marker for each class. Do not write the full solution. Explain how to select the first two columns of an array, which arguments of `plt.scatter()` set the coordinates and the marker of the points, and how to add axis labels.*

In [ ]:
# Assignment 1
def image_moments(region):
    """ Compute moments of the external contour in a binary image.   
    
        Args:
            region: Binary image
                    
        Returns: 
            moments: dictionary containing all moments of the region
    """   
    
    # Get external contour
    contours,_ = cv2.findContours(region,cv2.RETR_EXTERNAL ,cv2.CHAIN_APPROX_NONE)
    cnt = contours[0]
    
    # Compute moments
    moments = cv2.moments(cnt)
    
    return moments

In [ ]:
# Compute Hu moments for circle shape
hu_circles = np.zeros((20,7))

for i in range(20):
    img = cv2.imread(images_path + "circles/" + str(None) + ".png", 0)
    moments = image_moments(None)
    hu_circles[None,None] = cv2.HuMoments(None).flatten()

# Compute Hu moments for square shape
hu_squares = np.zeros((20,7))

for i in range(20):
    img = cv2.imread(images_path + "squares/" + str(None) + ".png", 0)
    moments = image_moments(None)
    hu_squares[None,None] = cv2.HuMoments(None).flatten()

# Compute Hu moments for triangle shape
hu_triangles = np.zeros((20,7))

for i in range(20):    
    img = cv2.imread(images_path + "triangles/" + str(None) + ".png", 0)
    moments = image_moments(None)
    hu_triangles[None,None] = cv2.HuMoments(None).flatten()

# Define plot axis
plt.axis([0.15, 0.2, -0.0005, 0.0015])

# Plot firsts two Hu moments
plt.xlabel("First Hu moment")
plt.ylabel("Second Hu moment")
plt.scatter(hu_triangles[None,None],hu_triangles[None,None],marker="^")
plt.scatter(hu_circles[None,None],hu_circles[None,None], marker="o")
plt.scatter(hu_squares[None,None],hu_squares[None,None], marker="s");

### Interlude

**You must save the computed Hu moments of the segmented data**, since we are going to need them in the next notebook. For saving NumPy matrices, you can use [`np.save()`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.save.html) (here we save them in the `./data/` folder).

In [ ]:
np.save("./data/hu_circles.npy",hu_circles)
np.save("./data/hu_triangles.npy",hu_triangles)
np.save("./data/hu_squares.npy",hu_squares)

## 7.1.1 Linear discriminant functions <a id="711"></a>

Taking a look at the scatter plot resultant from the previous assignment, we can see how the three types of signs could be segmented using 2 lines. For example: $\\[10pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/plot_linear.jpg" width="350"/>
    <figcaption>Fig. 5. First two moments of the binarized traffic signs and a possible division into regions.</figcaption>
</center>$\\[5pt]$

And that's what **linear discriminant functions** do! 

Let's start by describing how such lines could be represented. In the $x_1$-$x_2$ feature space, a line can be defined as:

$$w_1x_1 + w_2x_2 + b = 0$$

Note that we can generalize this 2D line to a hyperplane in a feature space of any dimension $N$:

$$w_1x_1 + w_2x_2 + \ldots + w_Nx_N + b = \mathbf{w}^T\mathbf{x} + b = 0$$

being $\mathbf{w} = [w_1, w_2, \ldots, w_N]^T$ the **weight vector** and $b$ the **bias**. In this way, we can divide the feature space for any dimension (e.g. $N = 7$ for Hu moments).

It is also common to use the **augmented representation**, which includes the bias in the weight vector and appends a 1 to the feature vector, both of dimension $N+1$:

$$\mathbf{w}^T\mathbf{x} + b = \tilde{\mathbf{w}}^T\tilde{\mathbf{x}}, \qquad \tilde{\mathbf{w}} = \begin{bmatrix}\mathbf{w}\\ b\end{bmatrix} = [w_1, \ldots, w_N, b]^T, \qquad \tilde{\mathbf{x}} = \begin{bmatrix}\mathbf{x}\\ 1\end{bmatrix} = [x_1, \ldots, x_N, 1]^T$$

### Working with two classes

If we are considering just two classes $C_1$ and $C_2$ (*e.g.* an object in a kitchen could be a spoon or a fork), we can define a linear discriminant function as:

$$d(\mathbf{x}) = \mathbf{w}^T\mathbf{x} + b$$

The weight vector $\mathbf{w}$ and the bias $b$ are computed automatically from a set of labeled data called **training data**, using techniques like the Perceptron, Logistic Regression or Support Vector Machines.

In this way, when we have a new object characterized by $\mathbf{x}$ to classify, $d(\mathbf{x})$ is computed and:
- if $d(\mathbf{x}) > 0$, the object is assigned to $C_1$,
- if $d(\mathbf{x}) < 0$, the object is assigned to $C_2$, and
- if $d(\mathbf{x}) = 0$, the object lies on the **decision boundary** between both classes.

That is, the linear discriminant function defines the decision boundary $d(\mathbf{x}) = 0$ between both classes, being $\mathbf{w}$ normal to it and $b$ controlling its offset. Fig. 6 illustrates this (blue dots represent characterized objects belonging to $C_1$, while red ones those belonging to $C_2$). $\\[6pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/linear_discriminant_2_classes.png" width="400"/>
    <figcaption>Fig. 6. A linear discriminant function $d(\mathbf{x})$ divides the feature space into two regions, separated by the decision boundary $d(\mathbf{x}) = 0$.</figcaption>
</center>$\\[5pt]$

### **<span style="color:green"><b><i>ASSIGNMENT 2. Classifying samples</i></b></span>**

Let's play a bit with a toy example of a linear discriminant function. Consider that the parameters learned during the training phase of the classifier are $\mathbf{w} = [1,0]^T$ and $b=-3$, so the linear discriminant function looks like $d(\mathbf{x}) = 1x_1 + 0x_2 -3$. In the augmented representation, $\tilde{\mathbf{w}} = [1,0,-3]^T$ and each sample is $\tilde{\mathbf{x}} = [x_1,x_2,1]^T$ (see the data matrix in the code cell below).

**Your task is to:** 

- classify some 2D data using the previous discriminant function (for that, use a combination of [`np.dot()`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html) and [`np.sign()`](https://numpy.org/doc/stable/reference/generated/numpy.sign.html)),  and
- plot them with different colors depending on their assigned class using [`plt.scatter()`](https://matplotlib.org/3.2.1/api/_as_gen/matplotlib.pyplot.scatter.html) and its `c` argument. 

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Evaluating a linear discriminant for many samples</font>  
> **Suggested prompt**  
> *I have a NumPy data matrix with one augmented feature vector [x1, x2, 1] per row, and an augmented weight vector as a column vector. Do not write the full solution. Explain how `np.dot()` computes the discriminant function of all the samples at once, which shape the result has, and what `np.sign()` returns for positive, negative and zero values.*

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Coloring points by class</font>  
> **Suggested prompt**  
> *In the previous assignment I used Matplotlib's `plt.scatter()` with a different marker for each class. Now I need to plot 2D points coloring each one according to its class (+1 or -1). Do not write the full solution. Explain how the `c` and `cmap` arguments work, and why the array passed to `c` may need `.flatten()`.*

In [ ]:
# Define augmented weight vector [w_1, w_2, b]
w = np.array([[1],[0],[-3]])

# Define data matrix (one augmented feature vector [x_1, x_2, 1] per row)
data = np.array([[2,2,1],[1,2,1],[2,1,1],[5,4,1],[4,5,1],[4,4,1]])

# Compute class for each data
classification = np.dot(None,None)
class_ = np.sign(None)

# Plot classified data
plt.scatter(None,None,c=None.flatten(), cmap="bwr")

# Plot discriminant line
x = np.linspace(3, 3, 1000)
y = np.linspace(0, 5, 1000)
plt.plot(x,y);

### Scenarios with $M$ classes

The previous idea can be generalized to an arbitrary number of classes $M$, defining a linear discriminant function for each class $C_k$, $k=1,\ldots,M$:

$$d_k(\mathbf{x}) = w_{k,1}x_1 + w_{k,2}x_2 + \ldots + w_{k,N}x_N + b_k = \mathbf{w}_k^T\mathbf{x} + b_k = \tilde{\mathbf{w}}_k^T\tilde{\mathbf{x}}$$

In this case, each discriminant function assigns a score to its class, and the object is assigned to the class whose discriminant function returns the highest value (see Fig. 7):

$$\hat{C}(\mathbf{x}) = \operatorname*{arg\,max}_k d_k(\mathbf{x})$$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/augmented_linear_discriminant.png" width="800"/>
    <figcaption>Fig. 7. Linear discriminant function of each class $C_k$, its augmented representation, and the classification rule.</figcaption>
</center>$\\[5pt]$

When more than 2 classes are considered, **the discriminant functions are no longer the decision boundaries between classes**. Instead, the decision boundary between two classes $C_i$ and $C_j$ is obtained where their discriminant scores are equal, that is, where:

$$d_{ij}(\mathbf{x}) = d_i(\mathbf{x})-d_j(\mathbf{x}) = (\tilde{\mathbf{w}}_i^T-\tilde{\mathbf{w}}_j^T)\,\tilde{\mathbf{x}} = \tilde{\mathbf{w}}_{ij}^T\tilde{\mathbf{x}} = 0$$

So:
- if $d_{ij}(\mathbf{x}) > 0$, $C_i$ is preferred to $C_j$,
- if $d_{ij}(\mathbf{x}) < 0$, $C_j$ is preferred to $C_i$, and
- $d_{ij}(\mathbf{x}) = 0$ is the boundary between $C_i$ and $C_j$.

Thereby, the decision region of each class is $R_i = \{\mathbf{x} : d_i(\mathbf{x}) > d_j(\mathbf{x}),\ \forall j \neq i\}$. Fig. 8 shows an example with 3 classes (notice that there is also a decision boundary between classes 1 and 3, $d_{13}(\mathbf{x}) = 0$, but it has been omitted for clarity).$\\[10pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/2lines-discriminant.png" width="350"/>
    <figcaption>Fig. 8. Decision boundaries between 3 categories.</figcaption>
</center>$\\[5pt]$

## 7.1.2 Separability <a id="712"></a>

An important concept regarding object recognition is such of **separability**. In this way, two sets of points defining two classes may be separable or not using a given set of features (e.g. $x_1$, $x_2$, or both).  

Two sets are **linearly separable** if it exists at least one line in the plane that leaves all the points belonging to one class on one side of the line and all those belonging to the second class on the other side. This idea immediately generalizes to higher-dimensional Euclidean spaces if the line is replaced by a hyperplane. 

Some illustrative examples of this:

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/separability_features.png" width="800"/>
    <figcaption>Fig. 9. Discriminative power of features: $x_1$ alone separates the classes, $x_2$ alone does not, and using both provides a clearer linear separation.</figcaption>
</center>$\\[5pt]$

- In this example, $x_1$ alone separates the classes with a simple threshold, while $x_2$ alone does not. We say that $x_1$ is a **discriminative** feature, that is, it provides valuable information for separating both classes. Using both features, the classes are also separable by a line, and the separation between them is even clearer.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/separability_new_feature.png" width="800"/>
    <figcaption>Fig. 10. Adding a new feature can help: classes that overlap in the $x_1$-$x_2$ space become separable using $x_3$.</figcaption>
</center>$\\[5pt]$

- In this example the classes overlap in the $x_1$-$x_2$ space, so they are not separable. However, the addition of a new feature $x_3$ can make them separable (e.g. with a threshold on $x_3$)!

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/separability_nonlinear.png" width="800"/>
    <figcaption>Fig. 11. Linear vs nonlinear separability: classes that are not linearly separable in the $x_1$-$x_2$ space can be separated by a nonlinear discriminant function.</figcaption>
</center>$\\[5pt]$

- This is an example of a non-linear separable problem. The classes can't be separated using a line, but a non-linear discriminant function (e.g. a curved boundary), also called **generalized discriminant function**, could do it. We will see this later in this notebook.

In short, separability depends on both the chosen features and the complexity of the discriminant function. However, adding features is not always the solution. **How many features ($N$) should we use?** As the dimension $N$ of the feature space grows, the space becomes increasingly sparse, and keeping the same density of training samples requires exponentially more data. This is known as the **curse of dimensionality**: the smaller the size of the training data, the fewer features should be used.

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> The curse of dimensionality</font>  
> **Suggested prompt**  
> *Explain the curse of dimensionality in classification with a simple numerical example, and which techniques are used in practice to reduce the number of features (e.g. feature selection or PCA).*

### Back to sign shape recognition

Back to our traffic sign shape recognition problem, **we are trying to separate** the usual shapes of traffic signs using the Hu moments. As a first approximation to this problem, we plotted the first and second Hu moments ($x_1$ and $x_2$) in order to analyze them. You should have obtained something like this:

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/separability-4.png" width="500"/>
    <figcaption>Fig. 12. Result of plotting the two first Hu moments computed from the images in our train data.</figcaption>
</center>$\\[5pt]$

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

**Now you are in a good position to answer these questions:**

- Is this problem linearly separable?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- Are the classes separable using only the first Hu moment ($x_1$)?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- Are the classes separable using only the second Hu moment ($x_2$)?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- Which dimension is more discriminative?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- Would be *AliquindoiCars* guys happy if we propose a solution for shape classification based on linear discriminant functions?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>    

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> Separability of the Hu moments</font>  
> **Suggested prompt**  
> *I have plotted the first two Hu moments of circular, rectangular and triangular traffic signs. Guide me, without giving me the answers, to reason whether the classes are linearly separable, which of the two features is more discriminative, and which problems a classifier based only on them could have with new images (e.g. noisy segmentations or perspective).*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

## **<span style="color:green"><b><i>OPTIONAL</i></b></span>** 7.1.3 Generalized discriminant function <a id="713"></a> 

As we saw in the previous section, some classes are not separable using a linear discriminant function. In those cases, we may need a **linear basis function model**, also called **generalized discriminant function**, which permit us to use non-linear functions to separate classes (notice that, despite of this, the problem may still not be separable).

The idea behind these discriminant functions is to transform the original feature space $\mathbf{x} \in \mathbb{R}^N$ into a new space $\mathbf{x}' = \mathbf{f}(\mathbf{x}) \in \mathbb{R}^{N'}$, usually of higher dimension ($N' > N$), through a set of **basis functions** $\mathbf{f}(\mathbf{x}) = [f_1(\mathbf{x}), f_2(\mathbf{x}), \ldots, f_{N'}(\mathbf{x})]^T$:$\\[5pt]$

$$d(\mathbf{x}) = w_1\ f_1(\mathbf{x}) + w_2\ f_2(\mathbf{x}) + \ldots + w_{N'}\ f_{N'}(\mathbf{x}) + b = \sum_{i=1}^{N'}w_i\ \underbrace{f_i(\mathbf{x})}_{\scriptsize\text{New space}} + b = \mathbf{w}^T \mathbf{f}(\mathbf{x}) + b$$

$$d(\mathbf{x}) = w_1\ x_1' + w_2\ x_2' + \ldots + w_{N'}\ x_{N'}' + b = \sum_{i=1}^{N'}w_i\ x_i' + b = \mathbf{w}^T \mathbf{x}' + b = d'(\mathbf{x}')$$

That is, the discriminant function is non-linear in $\mathbf{x}$, but **linear in the new space** $\mathbf{x}'$. Using the augmented representation:

$$
\left.\begin{aligned}
        \tilde{\mathbf{x}}' &= [f_1(\mathbf{x}) \ \ \ f_2(\mathbf{x}) \ \ \ \cdots \ \ \ f_{N'}(\mathbf{x}) \ \ \  1]^T\\
        \tilde{\mathbf{w}} &= [w_1 \ \ \ w_2 \ \ \ \cdots \ \ \ w_{N'} \ \ \ b]^T
       \end{aligned}
 \right\}
 \ \  d(\mathbf{x}) = \tilde{\mathbf{w}}^T \tilde{\mathbf{x}}'
$$
 
 being $\mathbf{f}(\mathbf{x})$ the basis functions!

### **<span style="color:green"><b><i>OPTIONAL 1: Transforming between spaces</i></b></span>**

Let's practice this transformation between spaces. **What to do?** You have to complete the method `transform_space()`, which transforms a set of 2D points into a new space defined by:

$$\tilde{\mathbf{x}}' = [x_1^2, \ x_1x_2, \ x_2^2, \ x_1, \ x_2, \ 1]^T$$

that is, $N'=5$ basis functions plus the 1 of the augmented representation.

The input set of points is a matrix with shape $(\text{n}\_\text{data}, 2)$, and the output is another matrix with shape $(\text{n}\_\text{data}, 6)$, being $\text{n}\_\text{data}$ the number of points in the set.

In [ ]:
# Optional 1
def transform_space(data_points):
    """ Transform a set of 2D points into a quadratic space.   
    
        Args:
            data_points: Input matrix containing a set of 2D points
                    
        Returns: 
            fx: set of points transformed to a quadratic space
    """  
    
    n_data = data_points.shape[None]
    
    fx = np.zeros((None,None))
    
    for i in range(n_data):
        
        x1 = data_points[None,None]
        x2 = data_points[None,None]
        
        fx[i,:] = [None, None, None, None, None, None]
        
    return fx

You can use next code to **test if the results are correct**:

In [ ]:
data = np.array([[2,3],[1,2],[2,7],[2,4],[1,5],[2,4]])

x = transform_space(data)

print(x)

<font color='blue'>**Expected output:**  </font>

     [[ 4.  6.  9.  2.  3.  1.]
     [ 1.  2.  4.  1.  2.  1.]
     [ 4. 14. 49.  2.  7.  1.]
     [ 4.  8. 16.  2.  4.  1.]
     [ 1.  5. 25.  1.  5.  1.]
     [ 4.  8. 16.  2.  4.  1.]]

### **<span style="color:green"><b><i>OPTIONAL 2: Classifying in the transformed space</i></b></span>**

Finally, we will see a toy example of a non-linear discriminant function in action. Concretely, we will consider a quadratic function, $d(\mathbf{x})=x_1^2-5x_2$, in a two-class problem with $\mathbf{x}=[x_1, x_2]^T$. Using the basis functions of the previous assignment ($N'=5$), the augmented feature vector in the new space is:

$$\tilde{\mathbf{x}}' = [x_1', \ldots, x_5', 1]^T = [x_1^2 \ \ x_1x_2 \ \ x_2^2 \ \ x_1 \ \ x_2 \ \ 1]^T$$

so:
- $x_1'=f_1(\mathbf{x})=x_1^2$,
- $x_2'=f_2(\mathbf{x})=x_1 x_2$,
- and so on.

and the augmented weight vector turns out to be:

$$\tilde{\mathbf{w}} = [w_1, \ldots, w_5, b]^T = [1, 0, 0, 0, -5, 0]^T$$

As for the new discriminant function we get:

$$d(\mathbf{x}) = \tilde{\mathbf{w}}^T\tilde{\mathbf{x}}' = w_1 x_1' + w_5 x_5'$$

**Your task is to classify some 2D data** using the previous discriminant function and plot them coloring each point with a color depending on its class. For that, you can rely on the [`np.dot()`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html), [`np.sign()`](https://numpy.org/doc/stable/reference/generated/numpy.sign.html)), and [`plt.scatter()`](https://matplotlib.org/3.2.1/api/_as_gen/matplotlib.pyplot.scatter.html) functions previously used in the second assignment. 

*Tip: Remember to transform the data space before applying the dot product.*

Expected results:

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/classification_in_the_transformed_space.png"/>
    <figcaption>Fig. 13. Classification of a number of samples in both, the original space and the transformed one.</figcaption>
</center>$\\[5pt]$

In [ ]:
# Optional 2

# Define augmented weight vector [w_1, ..., w_5, b]
w = np.array([[1],[0],[0],[0],[-5],[0]])

# Define data matrix
data = np.array([[2,0.5],[1.5,0.25],[3,1],[1,1],[1,0.5],[2,1],[5,5.3],[4,3],[3,2]])

# Transform from the x space to a f(x) space
transformed_data = transform_space(None)

# Compute class for each data
classification = np.dot(None,None)
class_ = np.sign(None)

# Plot classified data
matplotlib.rcParams['figure.figsize'] = (15.0, 15.0)
ax = plt.subplot(121)
ax.set_aspect('equal')
plt.title('Original space')
plt.xlabel("x_1")
plt.ylabel("x_2")
plt.axis([-0.5, 6.5, -0.5, 7.5])
plt.scatter(None,None,c=None.flatten(), cmap="bwr")

# Plot discriminant line
x = np.linspace(0, 6, 1000)
y = (w[0]*(x**2) + w[5])/-w[4]
plt.plot(x,y);

# Or equivalently plotting the data in the transformed space, 
# where the decision boundary defined by the discriminant function defines a line
ax = plt.subplot(122)
ax.set_aspect('equal')
plt.title('Transformed space')
plt.xlabel("x'_1")
plt.ylabel("x'_5")
plt.axis([-1, 26, -10, 15])
plt.scatter(None,None,c=None.flatten(), cmap="bwr")

# Plot discriminant line
x = np.linspace(0, 27, 1000)
y = (w[0]*(x) + w[5])/-w[4]
plt.plot(x,y);


## Conclusion

Well done! This was a mostly theoretical notebook, but it described the basis of object classification. You have learned so far:

- how linear and non-linear discriminant functions work,
- to classify objects in a scenario with an arbitrary number of belonging classes using multiple discriminant functions, and
- to check if a feature space allows for the definition of discriminative classifiers looking at the separability between classes.

In the following notebooks in this chapter we will see how to automatically retrieve the weight vector and the bias from a set of training data. Exciting, isn't it?

## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>